# Task 4 – AgriFresh Order Agent (LangChain + Groq + Google Sheets)
An agent that answers sample prices, store hours, and logs customer orders to a Google Sheet.

In [ ]:
!pip install -q langchain langchain-groq gspread

import os
from google.colab import userdata, auth
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.2, max_retries=3)

auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# Open the sheet if it exists, otherwise create it once with a header row
try:
    sh = gc.open("AgriFresh Orders")
    ws = sh.sheet1
except gspread.SpreadsheetNotFound:
    sh = gc.create("AgriFresh Orders")
    ws = sh.sheet1
    ws.append_row(["name", "product", "qty_kg", "pickup_date", "total"])
print("Sheet ready:", sh.url)

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Sample demonstration prices per kg (not live market rates)
PRODUCTS = {"rice": 60, "wheat": 35, "maize": 30, "onion": 40, "tomato": 30}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm",
         "sunday": "10 am to 6 pm"}   # sample demo hours

class Order(BaseModel):
    """An agricultural product order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    product: Literal["rice", "wheat", "maize", "onion", "tomato"] = Field(description="Product from the list")
    qty_kg: float = Field(description="Quantity in kg", gt=0)
    date: str = Field(description="Pickup date or time as the customer said it")

In [ ]:
@tool
def get_price(product: str, qty_kg: float = 1) -> str:
    """Get the total sample price in rupees for a quantity (kg) of one agricultural product.

    Args:
        product: One of "rice", "wheat", "maize", "onion" or "tomato"
        qty_kg: How many kg the customer wants
    """
    price = PRODUCTS.get(product.lower().strip())
    if price is None:
        return f"'{product}' is not listed. I'll check the latest rate with the seller. Listed: {', '.join(PRODUCTS)}."
    return f"{qty_kg} kg {product} = Rs {price * qty_kg:g} (sample price Rs {price}/kg, not a live market rate)"

@tool
def is_open(day: str) -> str:
    """Check the store's sample opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, product: str, qty_kg: float, date: str) -> str:
    """Save a confirmed order to the AgriFresh Orders sheet.
    Use only when the customer has given their name, product, quantity (kg) and pickup date or time."""
    try:
        order = Order(name=name, product=product.lower().strip(), qty_kg=qty_kg, date=date)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = PRODUCTS[order.product] * order.qty_kg
    ws.append_row([order.name, order.product, order.qty_kg, order.date, total])
    return f"Saved: {order.qty_kg:g} kg {order.product} for {order.name}, pickup {order.date}, Rs {total:g} (sample price)."

In [ ]:
from langchain.agents import create_agent

SYSTEM = """ROLE:   You are the friendly order assistant for AgriFresh, an agricultural products store.
FACTS:  Products: rice, wheat, maize, onion and tomato. Prices are sample demo rates per kg, not live market rates.
RULES:  Always use get_price for prices and is_open for timings; never guess them.
        Mention that prices are samples and may vary by market, quality, season and location.
        If a product or current rate is not listed, say "I'll check the latest rate with the seller."
        Never promise delivery time or place; orders are for pickup only.
        Before log_order, make sure you have name, product, quantity in kg and pickup date or time; ask if missing.
        After saving, read the order back with the total.
FORMAT: Under 60 words, friendly, English or Telugu matching the customer."""

agent = create_agent(model=llm, tools=[get_price, is_open, log_order], system_prompt=SYSTEM)

history = []
def chat(text: str) -> str:
    global history
    try:
        result = agent.invoke({"messages": history + [{"role": "user", "content": text}]})
        history = result["messages"]                  # keep the whole run as memory
        return history[-1].content
    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again in a minute."

print(chat("Are you open on friday? How much are 5 kg of rice?"))
print(chat("I'm Siddu. Please order 3 kg onion for 6 Dec at 10 am."))
print(chat("What products do you have?"))
print(chat("Do you have mangoes?"))